[DeepSeek-OCR: A Hands-On Guide With 7 Practical Examples](https://www.datacamp.com/tutorial/deepseek-ocr-hands-on-guide?utm_cid=19589720821&utm_aid=186331394349&utm_campaign=230119_1-ps-other~dsa-tofu~all_2-b2c_3-emea_4-prc_5-na_6-na_7-le_8-pdsh-go_9-nb-e_10-na_11-na&utm_loc=9193547-&utm_mtd=-c&utm_kw=&utm_source=google&utm_medium=paid_search&utm_content=ps-other~emea-en~dsa~tofu~tutorial~large-language-models&gad_source=1&gad_campaignid=19589720821&gbraid=0AAAAADQ9WsG03JVxHGkXYEPlR7lbQ1Juu&gclid=Cj0KCQjw77bPBhC_ARIsAGAjjV-iZkc0lnPP5XiWd2dgJQOLkh66wT9D7f6MGAgOBlryoHtpD9sNa9waApplEALw_wcB)


https://github.com/deepseek-ai/DeepSeek-OCR/tree/main

https://github.com/rednote-hilab/dots.ocr/tree/master

In [ ]:
!pip install -q "transformers==4.46.3" "tokenizers==0.20.3" einops addict easydict pillow gradio

In [ ]:
import os, time, torch, gradio as gr
from PIL import Image
from transformers import AutoModel, AutoTokenizer
from datetime import datetime
import random
import string

In [ ]:
model_id = "deepseek-ai/DeepSeek-OCR"

tok = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)

if tok.pad_token is None and tok.eos_token is not None:
    tok.pad_token = tok.eos_token

model = AutoModel.from_pretrained(
    model_id,
    trust_remote_code=True,
    use_safetensors=True,
    attn_implementation="eager"
).to(dtype=torch.bfloat16, device="cuda").eval()

In [ ]:
def new_run_dir(base="./runs"):
    os.makedirs(base, exist_ok=True)
    ts = datetime.now().strftime("%Y%m%d-%H%M%S")
    rid = ''.join(random.choices(string.ascii_lowercase + string.digits, k=5))
    path = os.path.join(base, f"run_{ts}_{rid}")
    os.makedirs(path)
    return path

def run_ocr(image, prompt):
    img = image.convert("RGB")

    # resize opcional
    if max(img.size) > 2000:
        scale = 2000 / max(img.size)
        img = img.resize((int(img.width * scale), int(img.height * scale)))

    run_dir = new_run_dir()
    img_path = os.path.join(run_dir, "input.png")
    img.save(img_path)

    with torch.inference_mode():
        result = model.infer(
            tok,
            prompt=prompt,
            image_file=img_path,
            output_path=run_dir,
            base_size=1024,
            image_size=640,
            crop_mode=False
        )

    return result

In [ ]:
PROMPTS = {
    "markdown": "<image>\n<|grounding|>Convert the document to markdown.",
    "plain_text": "<image>\nExtract all text.",
    "tables": "<image>\nParse all tables and return HTML.",
    "handwriting": "<image>\nExtract handwritten text."
}

# <image>
# Extract all text including handwritten fields.
# Preserve structure and identify totals, dates, and names.

In [ ]:
def process(image, mode):
    prompt = PROMPTS[mode]
    result = run_ocr(image, prompt)
    return result


with gr.Blocks() as demo:
    gr.Markdown("# DeepSeek OCR Demo")

    image_input = gr.Image(type="pil", label="Upload Image")

    mode = gr.Radio(
        ["markdown", "plain_text", "tables", "handwriting"],
        value="markdown",
        label="Mode"
    )

    output = gr.Textbox(label="Output", lines=20)

    btn = gr.Button("Run OCR")

    btn.click(process, inputs=[image_input, mode], outputs=output)

demo.launch()

In [ ]:
image = Image.open("recibo.png")

result = run_ocr(image, PROMPTS["plain_text"])

print(result)